# ResNet50 Regularized Run

This notebook runs the current low-risk regularized ResNet50 experiment for WikiArt artist classification. It keeps the staged fine-tuning setup, adds optimizer weight decay, uses stronger head dropout, and writes separate artifacts so the original notebook outputs are preserved.


In [ ]:
# --- Standard library imports --- #
import sys
from pathlib import Path

# --- Resolve project paths --- #
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "README.md").exists() and (candidate / "src").exists()
    ),
    cwd,
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = NOTEBOOKS_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# --- Local imports --- #
from metrics import classification
from models import resnet50
from utils import utils

# --- Third-party libraries --- #
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import (
    CSVLogger,
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.optimizers import Adam

print("Project root:", PROJECT_ROOT)
print("Notebook directory:", NOTEBOOKS_DIR)
print("Results directory:", RESULTS_DIR)


## MixUp Note

`apply_mixup` is intentionally not enabled in this notebook. The current training setup uses sparse integer labels together with `sparse_categorical_crossentropy` and `SparseMacroF1`, while MixUp produces soft labels and would require a categorical-loss-compatible pipeline.


In [ ]:
dataset_path = PROJECT_ROOT / "data"

train_dataset, validation_dataset, test_dataset = utils.load_image_datasets(
    dataset_path
)

class_names = train_dataset.class_names
num_classes = len(class_names)
class_to_idx = {name: i for i, name in enumerate(class_names)}

print("Classes:", num_classes)
print("Class mapping:", class_to_idx)

for images, labels in train_dataset.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Label dtype:", labels.dtype)


In [ ]:
def build_callbacks(csv_append):
    return [
        CSVLogger("./results/training_log_regularize.csv", append=csv_append),
        EarlyStopping(
            monitor="val_f1_score",
            mode="max",
            patience=5,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            mode="max",
            factor=0.2,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
        ModelCheckpoint(
            "./checkpoint/best_model_regularized.keras",
            monitor="val_macro_f1",
            mode="max",
            save_best_only=True,
        ),
    ]

In [ ]:
# --- Stage 1: train the classifier head with the backbone frozen --- #
model = resnet50.build_model(
    num_classes=num_classes, trainable_layers=0, dropout_rate=0.4, l2_strength=1e-4
)

model.compile(
    optimizer=Adam(learning_rate=1e-3, weight_decay=1e-4),
    loss="categorical_crossentropy",
    metrics=[F1Score(average="macro", threshold=None, name="f1_score")],
)

history_stage_1 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=8,
    callbacks=build_callbacks(csv_append=False),
)

In [ ]:
# --- Stage 2: fine-tune only the top 30 ResNet layers --- #
base_model = None
for layer in model.layers:
    if "resnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError("ResNet backbone not found inside the model.")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

trainable_backbone_layers = sum(1 for layer in base_model.layers if layer.trainable)
print("Trainable backbone layers:", trainable_backbone_layers)

model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[classification.SparseMacroF1(num_classes=num_classes)],
)

history_stage_2 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=build_callbacks(csv_append=True),
)


In [ ]:
history_dict = {
    "loss": history_stage_1.history["loss"] + history_stage_2.history["loss"],
    "val_loss": history_stage_1.history["val_loss"]
    + history_stage_2.history["val_loss"],
    "macro_f1": history_stage_1.history["macro_f1"]
    + history_stage_2.history["macro_f1"],
    "val_macro_f1": history_stage_1.history["val_macro_f1"]
    + history_stage_2.history["val_macro_f1"],
}

In [ ]:
epochs = range(1, len(history_dict["loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history_dict["loss"], "bo", label="Training loss")
plt.plot(epochs, history_dict["val_loss"], "b", label="Validation loss")
plt.title("Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history_dict["macro_f1"], "bo", label="Training macro_f1")
plt.plot(epochs, history_dict["val_macro_f1"], "b", label="Validation macro_f1")
plt.title("Macro F1")
plt.xlabel("Epochs")
plt.ylabel("Macro F1")
plt.legend()

plt.tight_layout()
plt.show()
